In [1]:
!pip install -q torch pandas numpy scikit-learn matplotlib shap openpyxl

In [2]:
import os, gc, json, random, warnings, platform, sys
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from google.colab import drive
drive.mount('/content/drive')
SEED=42
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
os.environ['PYTHONHASHSEED']=str(SEED)
random.seed(SEED);np.random.seed(SEED)
BASE_DIR=Path('/content/drive/MyDrive')
SALES_FILE=BASE_DIR/'model_ready_category_dataset.csv'
OUTPUT_DIR=BASE_DIR/'model_3_global_lstm_outputs'
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
DATE_COL='week_start';CATEGORY_COL='Product Category';TARGET_COL='weekly_quantity'
SEASONAL_PERIOD=52;MAX_HORIZON=12;FORECAST_HORIZONS=[4,8,12];EPS=1e-12
OUTER_ORIGINS={'spring_2025':'2025-04-07','summer_2025':'2025-07-07','early_autumn_2025':'2025-09-08','q4_year_end_2025':'2025-10-06'}
print('Sales dataset:',SALES_FILE,'Output:',OUTPUT_DIR)
assert SALES_FILE.exists(),f'Missing input: {SALES_FILE}'

Mounted at /content/drive
Sales dataset: /content/drive/MyDrive/model_ready_category_dataset.csv Output: /content/drive/MyDrive/model_3_global_lstm_outputs


In [3]:
import hashlib
raw_bytes=SALES_FILE.read_bytes()
sha256=hashlib.sha256(raw_bytes).hexdigest()
raw=pd.read_csv(SALES_FILE)
raw.columns=[c.strip() for c in raw.columns]
required=[CATEGORY_COL,DATE_COL,TARGET_COL]
if not set(required).issubset(raw):raise ValueError(f'Missing columns: {set(required)-set(raw)}')
raw=raw[required].copy()
raw[DATE_COL]=pd.to_datetime(raw[DATE_COL],errors='raise')
raw[TARGET_COL]=pd.to_numeric(raw[TARGET_COL],errors='raise')
if raw[required].isna().any().any():raise ValueError('Null values in input')
raw[CATEGORY_COL]=raw[CATEGORY_COL].astype(str).str.strip()
if (raw[TARGET_COL]<0).any():raise ValueError('Negative demand in input: resolve before running')
if raw.duplicated([CATEGORY_COL,DATE_COL]).any():raise ValueError('Duplicate category-week')
categories=sorted(raw[CATEGORY_COL].unique())
if len(categories)!=40:raise ValueError(f'Expected 40 categories, found {len(categories)}')
all_weeks=pd.date_range(raw[DATE_COL].min(),raw[DATE_COL].max(),freq='W-MON')
if len(all_weeks)==0 or raw[DATE_COL].dt.weekday.ne(0).any():raise ValueError('Expected Monday weeks')
idx=pd.MultiIndex.from_product([categories,all_weeks],names=[CATEGORY_COL,DATE_COL])
weekly=raw.set_index([CATEGORY_COL,DATE_COL]).reindex(idx).reset_index()
weekly[TARGET_COL]=weekly[TARGET_COL].fillna(0.0)
weekly=weekly.rename(columns={DATE_COL:'week',TARGET_COL:'target'})
weekly['series_id']=weekly[CATEGORY_COL].astype(str)
weekly=weekly.sort_values(['series_id','week']).reset_index(drop=True)
weekly['time_idx']=weekly.week.map({w:i for i,w in enumerate(all_weeks)}).astype(int)
weekly['month']=weekly.week.dt.month.astype(float)
weekly['week_of_year']=weekly.week.dt.isocalendar().week.astype(float)
weekly['week_sin']=np.sin(2*np.pi*weekly.week_of_year/52)
weekly['week_cos']=np.cos(2*np.pi*weekly.week_of_year/52)
weekly['month_sin']=np.sin(2*np.pi*weekly.month/12)
weekly['month_cos']=np.cos(2*np.pi*weekly.month/12)
weekly['seasonal_lag_52']=weekly.groupby('series_id').target.shift(52)
weekly['seasonal_lag_52_available']=weekly.seasonal_lag_52.notna().astype(float)
weekly['seasonal_lag_52']=weekly.seasonal_lag_52.fillna(0.0)
assert set(weekly.columns).issuperset({'target','seasonal_lag_52','week_sin','week_cos'})
assert len(weekly)==len(categories)*len(all_weeks)
weekly.to_csv(OUTPUT_DIR/'m3_lstm_panel.csv',index=False)
print('Categories',len(categories),'Weeks',len(all_weeks),'Panel rows',len(weekly),'Sales SHA256',sha256)
print('Range',all_weeks.min().date(),all_weeks.max().date())

Categories 40 Weeks 155 Panel rows 6200 Sales SHA256 b64da7b6cfe84735a155192e1b087298c76eba4e833577f3a7452b31eedb0788
Range 2023-01-09 2025-12-22


In [4]:
scenarios=[]
for label,origin_str in OUTER_ORIGINS.items():
 origin=pd.Timestamp(origin_str)
 for h in FORECAST_HORIZONS:
  dates=pd.date_range(origin,periods=h,freq='W-MON')
  if dates[-1]>weekly.week.max():raise ValueError('Test horizon exceeds panel')
  scenarios.append(dict(scenario_id=f'{label}_h{h}',origin_label=label,origin=origin_str,horizon_weeks=h,test_start=str(dates[0].date()),test_end=str(dates[-1].date())))
pd.DataFrame(scenarios).to_csv(OUTPUT_DIR/'m3_common_scenarios.csv',index=False)
def mase_scale_info(y_train,m=52):
 y=np.asarray(y_train,float)
 if len(y)<=m:return np.nan,'insufficient_history'
 d=float(np.mean(np.abs(y[m:]-y[:-m])))
 return (d,'valid') if np.isfinite(d) and d>EPS else (np.nan,'zero_or_invalid_scale')
print(pd.DataFrame(scenarios).to_string(index=False))

          scenario_id      origin_label     origin  horizon_weeks test_start   test_end
       spring_2025_h4       spring_2025 2025-04-07              4 2025-04-07 2025-04-28
       spring_2025_h8       spring_2025 2025-04-07              8 2025-04-07 2025-05-26
      spring_2025_h12       spring_2025 2025-04-07             12 2025-04-07 2025-06-23
       summer_2025_h4       summer_2025 2025-07-07              4 2025-07-07 2025-07-28
       summer_2025_h8       summer_2025 2025-07-07              8 2025-07-07 2025-08-25
      summer_2025_h12       summer_2025 2025-07-07             12 2025-07-07 2025-09-22
 early_autumn_2025_h4 early_autumn_2025 2025-09-08              4 2025-09-08 2025-09-29
 early_autumn_2025_h8 early_autumn_2025 2025-09-08              8 2025-09-08 2025-10-27
early_autumn_2025_h12 early_autumn_2025 2025-09-08             12 2025-09-08 2025-11-24
  q4_year_end_2025_h4  q4_year_end_2025 2025-10-06              4 2025-10-06 2025-10-27
  q4_year_end_2025_h8  q4_year_e

In [37]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import itertools, copy
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(SEED)
if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
try:
 torch.use_deterministic_algorithms(True)
except Exception as exc: print('Determinism setup warning:',exc)
if torch.cuda.is_available():
 torch.backends.cudnn.deterministic=True
 torch.backends.cudnn.benchmark=False
try: torch.set_float32_matmul_precision('highest')
except Exception: pass
VAL_ORIGINS=['2024-09-02','2024-11-04','2025-01-06']
LOOKBACK=52
H=12
MIN_HISTORY=65
COVARS=['week_sin','week_cos','month_sin','month_cos','seasonal_lag_52','seasonal_lag_52_available']
GRID=[dict(head=head,hidden=hidden,dropout=dropout,lr=lr,weight_decay=0.001,epochs=100,patience=14,batch_size=128,lookback=LOOKBACK,layers=3) for head,hidden,dropout,lr in itertools.product(['direct_huber','hurdle_huber'],[16],[0.1,0.25],[0.0005,0.001])]
print('Architecture: global three-layer LSTM with direct multi-horizon heads')
print('Device',DEVICE,'Candidates',len(GRID))

def arrays_at_origin(panel,cutoff,mode,lookback=LOOKBACK):
 cutoff=pd.Timestamp(cutoff)
 xs,cs,ys,ids,anchors,scales,seasonals=[],[],[],[],[],[],[]
 for ci,(cat,g) in enumerate(panel.groupby('series_id',sort=True)):
  g=g.sort_values('week').reset_index(drop=True)
  if mode=='forecast':
   indices=g.index[g.week.eq(cutoff)].tolist()
   if len(indices)!=1:raise ValueError(('Origin missing',cat,cutoff))
  else:indices=range(max(MIN_HISTORY,lookback),len(g)-H+1)
  for pos in indices:
   anchor=pd.Timestamp(g.at[pos,'week'])
   if mode=='train' and anchor+pd.Timedelta(weeks=H-1)>=cutoff:break
   past=g.iloc[pos-lookback:pos]
   future=g.iloc[pos:pos+H]
   if len(past)!=lookback or len(future)!=H:raise AssertionError('Incomplete window')
   if past.week.max()>=anchor:raise AssertionError('Historical leakage')
   if mode=='train' and future.week.max()>=cutoff:raise AssertionError('Training label leakage')
   if (future.week-pd.Timedelta(weeks=52)>=anchor).any():raise AssertionError('Future seasonal lag leakage')
   for lead in range(1,H+1):
    target=anchor+pd.Timedelta(weeks=lead-1)
    if target-pd.Timedelta(weeks=52)>=anchor:raise AssertionError('Seasonal leakage')
   scale=max(1.,float(past.target.mean()))
   x=np.log1p(past.target.to_numpy(float)/scale)
   cov=pd.concat([past,future],ignore_index=True)[COVARS].to_numpy(float)
   ix=COVARS.index('seasonal_lag_52')
   cov[:,ix]=np.log1p(np.maximum(cov[:,ix],0)/scale)
   baseline=future.seasonal_lag_52.to_numpy(float)
   xs.append(np.r_[x,np.log1p(scale)].astype('float32'))
   cs.append(cov.astype('float32'))
   ys.append(future.target.to_numpy('float32'))
   ids.append(ci);anchors.append(anchor);scales.append(scale);seasonals.append(baseline)
 if not xs:raise ValueError(('No samples',cutoff,mode))
 data=(np.stack(xs),np.stack(cs),np.stack(ys),np.array(ids,dtype='int64'),anchors,np.array(scales,dtype='float32'),np.stack(seasonals).astype('float32'))
 if not all(np.isfinite(a).all() for a in [data[0],data[1],data[2],data[5],data[6]]):raise ValueError('Nonfinite inputs')
 return data

class GlobalLSTM(nn.Module):
 def __init__(self,n_cov,n_series,hidden,dropout,head,lookback=LOOKBACK,horizon=H,layers=3):
  super().__init__()
  self.head=head;self.lookback=lookback;self.horizon=horizon
  self.embedding = nn.Embedding(n_series, 8)

  self.encoder=nn.LSTM(input_size=1+n_cov,hidden_size=hidden,num_layers=layers,batch_first=True,dropout=dropout if layers>1 else 0.0)
  self.future_proj=nn.Sequential(nn.Linear(horizon*n_cov,hidden),nn.GELU(),nn.Dropout(dropout))
  self.decoder = nn.Sequential(
    nn.Linear(hidden*2 + 8 + 1, hidden),
    nn.GELU(),
    nn.Dropout(dropout),
    nn.Linear(hidden, horizon*(2 if head=='hurdle_huber' else 1))
)
  self.skip=nn.Linear(lookback,horizon)
 def forward(self,x,c,sid):
  historical=torch.cat([x[:,:-1].unsqueeze(-1),c[:,:self.lookback,:]],dim=-1)
  _,(hn,_)=self.encoder(historical)
  future=self.future_proj(c[:,self.lookback:,:].flatten(1))
  z=self.decoder(torch.cat([hn[-1],future,self.embedding(sid),x[:,-1:]],dim=-1))
  z=z.reshape(-1,self.horizon,2 if self.head=='hurdle_huber' else 1)
  logmag=(z[...,0]+self.skip(x[:,:-1])).clamp(-8,12)
  if self.head=='hurdle_huber':return logmag,torch.sigmoid(z[...,1])
  return logmag,torch.ones_like(logmag)

def point_forecast(model,x,c,sid,scale):
 logmag,prob=model(x,c,sid)
 return (torch.expm1(F.softplus(logmag)).clamp_min(0)*prob*scale.unsqueeze(-1)).clamp(0,1e7)

def objective(model,x,c,sid,y,scale):
 logmag,prob=model(x,c,sid)
 yy=torch.log1p(y/scale.unsqueeze(-1))
 if model.head=='direct_huber':return F.smooth_l1_loss(F.softplus(logmag),yy,beta=0.5)
 positive=(y>0).float()
 occurrence=F.binary_cross_entropy(prob.clamp(1e-6,1-1e-6),positive)
 magnitude=F.smooth_l1_loss(F.softplus(logmag),yy,reduction='none',beta=0.5)
 magnitude=(magnitude*positive).sum()/positive.sum().clamp_min(1.)
 return occurrence+magnitude

def fit_lstm(train,val,params,seed=SEED):
 torch.manual_seed(seed)
 model=GlobalLSTM(len(COVARS),len(categories),params['hidden'],params['dropout'],params['head'],params['lookback'],layers=params['layers']).to(DEVICE)
 opt=torch.optim.AdamW(model.parameters(),lr=params['lr'],weight_decay=params['weight_decay'])
 tr=[torch.from_numpy(a) for a in (train[0],train[1],train[2],train[3],train[5])]
 va=[torch.from_numpy(a).to(DEVICE) for a in (val[0],val[1],val[2],val[3],val[5])]
 loader=DataLoader(TensorDataset(*tr),batch_size=params['batch_size'],shuffle=True,generator=torch.Generator().manual_seed(seed))
 best=float('inf');state=None;best_epoch=0;bad=0;history=[]
 for epoch in range(1,params['epochs']+1):
  model.train();total=0.;count=0
  for batch in loader:
   x,c,y,sid,scale=[a.to(DEVICE) for a in batch]
   opt.zero_grad(set_to_none=True)
   loss=objective(model,x,c,sid,y,scale)
   if not torch.isfinite(loss):raise FloatingPointError(('Nonfinite loss',params,epoch))
   loss.backward();nn.utils.clip_grad_norm_(model.parameters(),1.);opt.step()
   total+=float(loss.detach().cpu())*len(x);count+=len(x)
  model.eval()
  with torch.no_grad():
   vx,vc,vy,vs,vscale=va
   vl=float(objective(model,vx,vc,vs,vy,vscale).cpu())
   vp=point_forecast(model,vx,vc,vs,vscale)
   vmae=float((vp-vy).abs().mean().cpu())
  history.append(dict(epoch=epoch,train_loss=total/count,val_loss=vl,val_MAE=vmae))
  if vl<best-1e-5:best=vl;state=copy.deepcopy(model.state_dict());best_epoch=epoch;bad=0
  else:bad+=1
  if bad>=params['patience']:break
 if state is None:raise RuntimeError('No finite validation state')
 model.load_state_dict(state)
 return model,pd.DataFrame(history),best_epoch

def predict_lstm(model,data):
 model.eval();pred=[]
 with torch.no_grad():
  for a in range(0,len(data[0]),256):
   x=torch.from_numpy(data[0][a:a+256]).to(DEVICE)
   c=torch.from_numpy(data[1][a:a+256]).to(DEVICE)
   sid=torch.from_numpy(data[3][a:a+256]).to(DEVICE)
   scale=torch.from_numpy(data[5][a:a+256]).to(DEVICE)
   pred.append(point_forecast(model,x,c,sid,scale).cpu().numpy())
 return np.concatenate(pred)

def evaluate_predictions(pred,data,panel,origin,label):
 rows=[]
 for j,ci in enumerate(data[3]):
  cat=categories[int(ci)]
  hist=panel.loc[(panel.series_id==cat)&(panel.week<pd.Timestamp(origin))].sort_values('week').target.to_numpy(float)
  den,status=mase_scale_info(hist,52)
  for h in FORECAST_HORIZONS:
   a=data[2][j,:h].astype(float);p=pred[j,:h].astype(float)
   nz=np.abs(a)>EPS;sm=np.abs(a)+np.abs(p)
   rows.append(dict(origin_label=label,origin=str(pd.Timestamp(origin).date()),category=cat,horizon_weeks=h,MASE=float(np.mean(np.abs(a-p))/den) if status=='valid' else np.nan,MAE=float(np.mean(np.abs(a-p))),RMSE=float(np.sqrt(np.mean((a-p)**2))),MAPE=float(np.mean(np.abs((a[nz]-p[nz])/a[nz]))*100) if nz.any() else np.nan,sMAPE=float(np.mean(np.divide(2*np.abs(a-p),sm,out=np.zeros_like(sm),where=sm>EPS))*100),mase_status=status))
 return pd.DataFrame(rows)


Architecture: global three-layer LSTM with direct multi-horizon heads
Device cuda Candidates 8


In [38]:
assert len(categories) == 40
assert LOOKBACK == 52 and H == 12 and FORECAST_HORIZONS == [4, 8, 12]
assert VAL_ORIGINS == ['2024-09-02', '2024-11-04', '2025-01-06']
assert list(OUTER_ORIGINS.values()) == ['2025-04-07', '2025-07-07', '2025-09-08', '2025-10-06']
assert max(pd.Timestamp(v) + pd.Timedelta(weeks=H-1) for v in VAL_ORIGINS) < min(pd.Timestamp(v) for v in OUTER_ORIGINS.values())
for origin_text in list(VAL_ORIGINS) + list(OUTER_ORIGINS.values()):
    cutoff = pd.Timestamp(origin_text)
    tr = arrays_at_origin(weekly, cutoff, 'train')
    assert all(anchor + pd.Timedelta(weeks=H-1) < cutoff for anchor in tr[4])
    te = arrays_at_origin(weekly, cutoff, 'forecast')
    assert all(anchor == cutoff for anchor in te[4])
    assert len(te[0]) == len(categories)
    assert np.isfinite(te[0]).all() and np.isfinite(te[1]).all()
print('PASS: training labels strictly precede each origin; validation finishes before outer test.')
print('NOTE: Missing category-weeks are assumed to mean zero demand; verify this against source data.')
print('NOTE: Prior inspection of 2025 test scores makes subsequent model choice exploratory, not a fresh holdout.')

PASS: training labels strictly precede each origin; validation finishes before outer test.
NOTE: Missing category-weeks are assumed to mean zero demand; verify this against source data.
NOTE: Prior inspection of 2025 test scores makes subsequent model choice exploratory, not a fresh holdout.


In [39]:
validation=[];curves=[]
for val_str in VAL_ORIGINS:
 origin=pd.Timestamp(val_str)
 panel=weekly
 tr=arrays_at_origin(panel,origin,'train')
 va=arrays_at_origin(panel,origin,'forecast')
 assert len(va[0])==len(categories)
 assert max(tr[4])+pd.Timedelta(weeks=H-1)<origin
 print('INNER',val_str,'train',len(tr[0]),'validation',len(va[0]))
 for config_id,params in enumerate(GRID):
  try:
   model,hist,ep=fit_lstm(tr,va,params)
   pred=predict_lstm(model,va)
   metrics=evaluate_predictions(pred,va,panel,origin,val_str)
   validation.append(dict(validation_origin=val_str,config_id=config_id,**params,best_epoch=ep,mean_MASE=float(metrics.MASE.mean()),mean_MAE=float(metrics.MAE.mean()),valid_MASE=int(metrics.MASE.notna().sum()),status='ok'))
   curves.append(hist.assign(validation_origin=val_str,config_id=config_id))
   print(config_id,params['head'],params['hidden'],params['dropout'],'MASE',round(metrics.MASE.mean(),5),'epoch',ep)
   del model
  except (FloatingPointError,RuntimeError) as exc:
   validation.append(dict(validation_origin=val_str,config_id=config_id,**params,best_epoch=np.nan,mean_MASE=np.nan,mean_MAE=np.nan,valid_MASE=0,status=str(exc)))
   print('FAILED',config_id,str(exc))
  if torch.cuda.is_available():torch.cuda.empty_cache()
val_df=pd.DataFrame(validation)
val_df.to_csv(OUTPUT_DIR/'m3_inner_all_trials.csv',index=False)
if curves:pd.concat(curves,ignore_index=True).to_csv(OUTPUT_DIR/'m3_inner_learning_curves.csv',index=False)
rank=val_df.loc[val_df.status.eq('ok')].groupby('config_id').agg(mean_MASE=('mean_MASE','mean'),worst_MASE=('mean_MASE','max'),mean_MAE=('mean_MAE','mean'),n_origins=('mean_MASE','count')).reset_index()
rank=rank.loc[rank.n_origins.eq(len(VAL_ORIGINS))].sort_values(['mean_MASE','worst_MASE','mean_MAE','config_id'])
if rank.empty:raise RuntimeError('No configuration succeeded at every validation origin')
selected_id=int(rank.iloc[0].config_id);selected=GRID[selected_id]
selected_epochs=max(1,int(round(val_df.loc[val_df.config_id.eq(selected_id),'best_epoch'].median())))
print('SELECTED',selected_id,selected,'epochs',selected_epochs)
display(rank);display(val_df)
with open(OUTPUT_DIR/'m3_selected_hyperparameters.json','w') as f:json.dump(dict(config_id=selected_id,params=selected,epochs=selected_epochs,origins=VAL_ORIGINS,criterion='mean inner origin MASE'),f,indent=2)


INNER 2024-09-02 train 400 validation 40
0 direct_huber 16 0.1 MASE 9.68716 epoch 48
1 direct_huber 16 0.1 MASE 9.68149 epoch 30
2 direct_huber 16 0.25 MASE 9.63411 epoch 50
3 direct_huber 16 0.25 MASE 9.63991 epoch 31
4 hurdle_huber 16 0.1 MASE 10.03736 epoch 28
5 hurdle_huber 16 0.1 MASE 9.9671 epoch 16
6 hurdle_huber 16 0.25 MASE 10.06053 epoch 28
7 hurdle_huber 16 0.25 MASE 9.98832 epoch 16
INNER 2024-11-04 train 760 validation 40
0 direct_huber 16 0.1 MASE 0.53366 epoch 25
1 direct_huber 16 0.1 MASE 0.51931 epoch 13
2 direct_huber 16 0.25 MASE 0.4627 epoch 37
3 direct_huber 16 0.25 MASE 0.38129 epoch 28
4 hurdle_huber 16 0.1 MASE 0.44718 epoch 33
5 hurdle_huber 16 0.1 MASE 0.43258 epoch 22
6 hurdle_huber 16 0.25 MASE 0.39984 epoch 55
7 hurdle_huber 16 0.25 MASE 0.40636 epoch 24
INNER 2025-01-06 train 1120 validation 40
0 direct_huber 16 0.1 MASE 0.90317 epoch 15
1 direct_huber 16 0.1 MASE 0.80874 epoch 20
2 direct_huber 16 0.25 MASE 0.87876 epoch 16
3 direct_huber 16 0.25 MASE 0.8

,config_id,mean_MASE,worst_MASE,mean_MAE,n_origins
3,3,3.612531,9.639910,63.728439,3
2,2,3.658527,9.634112,66.057513,3
1,1,3.669847,9.681494,63.964841,3
0,0,3.707998,9.687165,65.989211,3
5,5,3.763309,9.967099,68.019811,3
7,7,3.764022,9.988316,67.277077,3
6,6,3.775549,10.060532,67.471569,3
4,4,3.790543,10.037356,68.201485,3


,validation_origin,config_id,head,hidden,dropout,lr,weight_decay,epochs,patience,batch_size,lookback,layers,best_epoch,mean_MASE,mean_MAE,valid_MASE,status
0,2024-09-02,0,direct_huber,16,0.10,0.0005,0.001,100,14,128,52,3,48,9.687165,122.532748,111,ok
1,2024-09-02,1,direct_huber,16,0.10,0.0010,0.001,100,14,128,52,3,30,9.681494,122.516838,111,ok
2,2024-09-02,2,direct_huber,16,0.25,0.0005,0.001,100,14,128,52,3,50,9.634112,122.048124,111,ok
3,2024-09-02,3,direct_huber,16,0.25,0.0010,0.001,100,14,128,52,3,31,9.639910,122.755512,111,ok
4,2024-09-02,4,hurdle_huber,16,0.10,0.0005,0.001,100,14,128,52,3,28,10.037356,130.635476,111,ok
5,2024-09-02,5,hurdle_huber,16,0.10,0.0010,0.001,100,14,128,52,3,16,9.967099,129.606667,111,ok
6,2024-09-02,6,hurdle_huber,16,0.25,0.0005,0.001,100,14,128,52,3,28,10.060532,131.317601,111,ok
7,2024-09-02,7,hurdle_huber,16,0.25,0.0010,0.001,100,14,128,52,3,16,9.988316,130.126165,111,ok
8,2024-11-04,0,direct_huber,16,0.10,0.0005,0.001,100,14,128,52,3,25,0.533663,25.651300,117,ok
9,2024-11-04,1,direct_huber,16,0.10,0.0010,0.001,100,14,128,52,3,13,0.519310,22.896315,117,ok


In [40]:
def fit_final(train,params,epochs):
 torch.manual_seed(SEED)
 model=GlobalLSTM(len(COVARS),len(categories),params['hidden'],params['dropout'],params['head'],params['lookback'],layers=params['layers']).to(DEVICE)
 opt=torch.optim.AdamW(model.parameters(),lr=params['lr'],weight_decay=params['weight_decay'])
 tr=[torch.from_numpy(a) for a in (train[0],train[1],train[2],train[3],train[5])]
 loader=DataLoader(TensorDataset(*tr),batch_size=params['batch_size'],shuffle=True,generator=torch.Generator().manual_seed(SEED))
 logs=[]
 for epoch in range(1,epochs+1):
  model.train();total=0.;count=0
  for batch in loader:
   x,c,y,sid,scale=[a.to(DEVICE) for a in batch]
   opt.zero_grad(set_to_none=True);loss=objective(model,x,c,sid,y,scale)
   if not torch.isfinite(loss):raise FloatingPointError('Nonfinite final loss')
   loss.backward();nn.utils.clip_grad_norm_(model.parameters(),1.);opt.step()
   total+=float(loss.detach().cpu())*len(x);count+=len(x)
  logs.append(dict(epoch=epoch,train_loss=total/count))
 return model,pd.DataFrame(logs)

outer=[];forecasts=[];train_curves=[]
for label,origin_str in OUTER_ORIGINS.items():
 origin=pd.Timestamp(origin_str);panel=weekly
 tr=arrays_at_origin(panel,origin,'train');test=arrays_at_origin(panel,origin,'forecast')
 assert max(tr[4])+pd.Timedelta(weeks=H-1)<origin
 model,logs=fit_final(tr,selected,selected_epochs)
 pred=predict_lstm(model,test)
 assert pred.shape==(len(categories),H) and np.isfinite(pred).all()
 metrics=evaluate_predictions(pred,test,panel,origin,label)
 outer.append(metrics);train_curves.append(logs.assign(origin_label=label))
 for j,ci in enumerate(test[3]):
  for lead in range(H):forecasts.append(dict(origin_label=label,origin=origin_str,category=categories[int(ci)],week=str((origin+pd.Timedelta(weeks=lead)).date()),lead_step=lead+1,actual=float(test[2][j,lead]),prediction=float(pred[j,lead])))
 print('OUTER',label,'MASE',round(metrics.MASE.mean(),6))
 del model
 if torch.cuda.is_available():torch.cuda.empty_cache()
results=pd.concat(outer,ignore_index=True)
assert len(results)==480
assert results.MASE.notna().sum()==468, 'Unexpected MASE validity; inspect input data before comparison'
results.to_csv(OUTPUT_DIR/'m3_outer_category_origin_horizon_metrics.csv',index=False)
pd.DataFrame(forecasts).to_csv(OUTPUT_DIR/'m3_outer_weekly_forecasts.csv',index=False)
pd.concat(train_curves,ignore_index=True).to_csv(OUTPUT_DIR/'m3_outer_train_curves.csv',index=False)
summary=results[['MASE','MAE','RMSE','MAPE','sMAPE']].mean()
by_horizon=results.groupby('horizon_weeks')[['MASE','MAE','RMSE','MAPE','sMAPE']].mean()
summary.to_csv(OUTPUT_DIR/'m3_overall_summary.csv');by_horizon.to_csv(OUTPUT_DIR/'m3_by_horizon.csv')
print('OVERALL SUMMARY');print(summary.to_string());print('Valid MASE',results.MASE.notna().sum(),'undefined',results.MASE.isna().sum())
print('BY HORIZON');print(by_horizon.to_string())

results.groupby('origin_label')[['MASE','MAE','RMSE','MAPE','sMAPE']].mean().to_csv(OUTPUT_DIR/'m3_by_origin.csv')
results.groupby(['origin_label','horizon_weeks'])[['MASE','MAE','RMSE','MAPE','sMAPE']].mean().to_csv(OUTPUT_DIR/'m3_by_origin_horizon.csv')
results.groupby('category')[['MASE','MAE','RMSE','MAPE','sMAPE']].mean().to_csv(OUTPUT_DIR/'m3_by_category.csv')


OUTER spring_2025 MASE 0.755088
OUTER summer_2025 MASE 0.510605
OUTER early_autumn_2025 MASE 1.615343
OUTER q4_year_end_2025 MASE 0.459934
OVERALL SUMMARY
MASE       0.835242
MAE       52.962410
RMSE      68.398786
MAPE     121.699307
sMAPE    141.319356
Valid MASE 468 undefined 12
BY HORIZON
                   MASE        MAE       RMSE        MAPE       sMAPE
horizon_weeks                                                        
4              0.976548  60.611219  71.631030  108.962693  137.060751
8              0.763184  49.401239  64.420759  110.301672  141.412818
12             0.765995  48.874771  69.144569  142.858161  145.484500


In [41]:
import shap
SHAP_DIR=OUTPUT_DIR/'shap'
SHAP_DIR.mkdir(parents=True,exist_ok=True)
SHAP_ORIGIN='q4_year_end_2025'
SHAP_HORIZONS=[4,8,12]
SHAP_BACKGROUND=24
SHAP_EXAMPLES=20
shap_origin=pd.Timestamp(OUTER_ORIGINS[SHAP_ORIGIN]);shap_panel=weekly
shap_train=arrays_at_origin(shap_panel,shap_origin,'train')
shap_test=arrays_at_origin(shap_panel,shap_origin,'forecast')
shap_model,_=fit_final(shap_train,selected,selected_epochs)
shap_model=shap_model.cpu().eval()
SHAP_DEVICE=torch.device("cpu")
feature_names=['past_demand','historical_scale']+COVARS
shap_rows=[];shap_checks=[]
class SHAPWrapper(nn.Module):
 def __init__(self,model,sid,scale,horizon):
  super().__init__();self.model=model;self.sid=int(sid);self.scale=float(scale);self.horizon=int(horizon)
 def forward(self,flat):
  x=flat[:,:LOOKBACK+1]
  c=flat[:,LOOKBACK+1:].reshape(-1,LOOKBACK+H,len(COVARS))
  sid=torch.full((len(flat),),self.sid,device=flat.device,dtype=torch.long)
  scale=torch.full((len(flat),),self.scale,device=flat.device,dtype=flat.dtype)
  return point_forecast(self.model,x,c,sid,scale)[:,:self.horizon].mean(dim=1,keepdim=True)
for j in range(min(SHAP_EXAMPLES,len(shap_test[0]))):
 sid=int(shap_test[3][j]);category=categories[sid]
 idx=np.flatnonzero(shap_train[3]==sid)
 if len(idx)<2:continue
 rng=np.random.default_rng(SEED+j)
 bgidx=rng.choice(idx,size=min(SHAP_BACKGROUND,len(idx)),replace=False)
 background=np.concatenate([shap_train[0][bgidx],shap_train[1][bgidx].reshape(len(bgidx),-1)],axis=1)
 sample=np.concatenate([shap_test[0][j:j+1],shap_test[1][j:j+1].reshape(1,-1)],axis=1)
 for h in SHAP_HORIZONS:
  wrapper=SHAPWrapper(shap_model,sid,shap_test[5][j],h).to(SHAP_DEVICE).eval()
  explainer=shap.GradientExplainer(wrapper,torch.tensor(background,device=SHAP_DEVICE))
  values=explainer.shap_values(torch.tensor(sample,device=SHAP_DEVICE),nsamples=80)
  if isinstance(values,list):values=values[0]
  arr=np.asarray(values).reshape(-1)
  expected=(wrapper(torch.tensor(background,device=SHAP_DEVICE)).detach().cpu().numpy().mean())
  predicted=float(wrapper(torch.tensor(sample,device=SHAP_DEVICE)).detach().cpu().item())
  shap_checks.append(dict(category=category,horizon_weeks=h,baseline=float(expected),prediction=predicted,shap_sum=float(arr.sum()),approx_error=float(predicted-expected-arr.sum())))
  shap_rows.append(dict(category=category,horizon_weeks=h,feature='past_demand',feature_value=float(shap_test[0][j,:LOOKBACK].mean()),shap_value=float(arr[:LOOKBACK].sum())))
  shap_rows.append(dict(category=category,horizon_weeks=h,feature='historical_scale',feature_value=float(shap_test[0][j,LOOKBACK]),shap_value=float(arr[LOOKBACK])))
  cv=arr[LOOKBACK+1:].reshape(LOOKBACK+H,len(COVARS))
  for k,name in enumerate(COVARS):
   shap_rows.append(dict(category=category,horizon_weeks=h,feature=name,feature_value=float(shap_test[1][j,:,k].mean()),shap_value=float(cv[:,k].sum())))
 print('SHAP',j+1,'/',min(SHAP_EXAMPLES,len(shap_test[0])),category)
shap_df=pd.DataFrame(shap_rows);checks=pd.DataFrame(shap_checks)

grouped_sums=shap_df.groupby(['category','horizon_weeks'])['shap_value'].sum().rename('grouped_shap_sum')
checks=checks.merge(grouped_sums.reset_index(),on=['category','horizon_weeks'],validate='one_to_one')
if not np.allclose(checks['shap_sum'],checks['grouped_shap_sum'],rtol=1e-5,atol=1e-3):
 raise AssertionError('Grouped SHAP contributions omit or duplicate input dimensions')

print('SHAP absolute approximation error (median / max):',
      checks.approx_error.abs().median(),checks.approx_error.abs().max())
shap_df.to_csv(SHAP_DIR/'m3_shap_grouped_values.csv',index=False)
checks.to_csv(SHAP_DIR/'m3_shap_approximation_checks.csv',index=False)
for h in SHAP_HORIZONS:
 frame=shap_df.loc[shap_df.horizon_weeks.eq(h)]
 matrix=frame.pivot(index='category',columns='feature',values='shap_value').reindex(columns=feature_names)
 vals=frame.pivot(index='category',columns='feature',values='feature_value').reindex(index=matrix.index,columns=feature_names)
 importance=matrix.abs().mean().sort_values(ascending=False)
 importance.to_csv(SHAP_DIR/f'm3_importance_h{h}.csv')
 plt.figure(figsize=(10,6));importance.iloc[:16].sort_values().plot.barh();plt.tight_layout();plt.savefig(SHAP_DIR/f'm3_global_importance_h{h}.png',dpi=160);plt.close()
 explanation=shap.Explanation(values=matrix.to_numpy(),data=vals.to_numpy(),feature_names=feature_names)
 shap.plots.beeswarm(explanation,max_display=16,show=False);plt.tight_layout();plt.savefig(SHAP_DIR/f'm3_beeswarm_h{h}.png',dpi=160,bbox_inches='tight');plt.close()
 for name in ['seasonal_lag_52', 'week_sin', 'week_cos', 'month_sin', 'month_cos']:
  plt.figure(figsize=(7,5));plt.scatter(vals[name],matrix[name],alpha=0.7);plt.xlabel(name);plt.ylabel('SHAP');plt.tight_layout();plt.savefig(SHAP_DIR/f'm3_dependence_{name}_h{h}.png',dpi=160);plt.close()
 cat=matrix.index[0];base=float(checks.loc[checks.category.eq(cat)&checks.horizon_weeks.eq(h),'baseline'].iloc[0])
 water=shap.Explanation(values=matrix.loc[cat].to_numpy(),base_values=base,data=vals.loc[cat].to_numpy(),feature_names=feature_names)
 shap.plots.waterfall(water,max_display=16,show=False);plt.tight_layout();plt.savefig(SHAP_DIR/f'm3_waterfall_h{h}.png',dpi=160,bbox_inches='tight');plt.close()
print('SHAP complete',len(checks),'explanations')
print('SHAP gradient estimates are approximate; check m3_shap_approximation_checks.csv')
print('Interpretation: historical_scale is included in SHAP grouping. The SHAP wrapper holds category and output scale fixed; attributions are conditional on those fixed quantities.')

SHAP 1 / 20 AΜΠΙΓΙΕ
SHAP 2 / 20 POUCH
SHAP 3 / 20 ΑΝΔΡΙΚΗ ΤΣΑΝΤΑ
SHAP 4 / 20 ΓΑΝΤΙ ΑΝΔΡΙΚΟ
SHAP 5 / 20 ΓΑΝΤΙ ΓΥΝΑΙΚΕΙΟ
SHAP 6 / 20 ΓΟΥΝΑ
SHAP 7 / 20 ΕΣΑΡΠΑ
SHAP 8 / 20 ΖΩΝΗ ΑΝΔΡΙΚΗ
SHAP 9 / 20 ΖΩΝΗ ΓΥΝΑΙΚΕΙΑ
SHAP 10 / 20 ΙΜΑΝΤΑΣ
SHAP 11 / 20 ΚAΠΕΛΟ
SHAP 12 / 20 ΚAΣΚΟΛ
SHAP 13 / 20 ΚΑΠΑ
SHAP 14 / 20 ΚΑΣΚΟΛ ΑΝΔΡΙΚΟ
SHAP 15 / 20 ΚΑΦΤΑΝΙ
SHAP 16 / 20 ΚΡΕΜΑΣΤΡΑ
SHAP 17 / 20 ΚΡΟΠ ΤΟΠ
SHAP 18 / 20 ΛΑΙΜΟΣ
SHAP 19 / 20 ΜΠΛΟΥΖΑ
SHAP 20 / 20 ΜΠΡΕΛΟΚ
SHAP absolute approximation error (median / max): 0.7879258274271153 266.1518763599877
SHAP complete 60 explanations
SHAP gradient estimates are approximate; check m3_shap_approximation_checks.csv
Interpretation: historical_scale is included in SHAP grouping. The SHAP wrapper holds category and output scale fixed; attributions are conditional on those fixed quantities.


In [42]:
import importlib.metadata as im
versions={p:im.version(p) for p in ['torch','numpy','pandas','scikit-learn','shap','matplotlib']}
manifest=dict(seed=SEED,python=sys.version,platform=platform.platform(),torch_cuda=torch.version.cuda,device=str(DEVICE),versions=versions,sales_sha256=sha256,features=COVARS,lookback=LOOKBACK,horizon=H,inner_origins=VAL_ORIGINS,outer_origins=OUTER_ORIGINS,selected_params=selected,selected_epochs=selected_epochs,architecture="global_lstm_3layer_embedding8_skip")
(OUTPUT_DIR/'m3_run_manifest.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False,default=str))
print('Run manifest saved.')

Run manifest saved.
